# 부산광역시 2026년 9월 생활인구 / 주민등록인구 비율

행정동별 생활인구 비율을 5개 고정 구간으로 나눈 단계구분도입니다. 생활인구는 시간대별 주거·직장·방문인구 합계의 24시간 평균, 주민등록인구는 2026년 9월 말 전체 당월 인구를 사용합니다. 시간대 값을 합산하지 않아 하루 인구가 중복 계산되지 않도록 했습니다.

- 생활인구: [부산광역시 OpenAPI](https://www.data.go.kr/data/15164444/openapi.do), 기준년월 202609
- 주민등록인구: [행정안전부 CSV](https://www.data.go.kr/data/3033254/fileData.do), 기준일 2026-09-30
- 경계: 2025-01-01 행정동 GeoJSON에서 부산 206개 동 추출
- 배경지도: V-World WMTS (키는 상위 폴더의 .env에서 읽고, 지도 HTML/키는 저장소에 포함하지 않음)


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from dotenv import load_dotenv
from IPython.display import IFrame, display

# 노트북을 저장소 폴더 또는 상위 작업 폴더에서 실행해도 data/를 찾습니다.
cwd = Path.cwd()
if (cwd / "data" / "busan_living_population_202609.csv").exists():
    project_dir = cwd
elif (cwd / "shrinking-busan" / "data" / "busan_living_population_202609.csv").exists():
    project_dir = cwd / "shrinking-busan"
else:
    raise FileNotFoundError("shrinking-busan/data 파일을 찾을 수 없습니다.")
data_dir = project_dir / "data"

# 현재 폴더 또는 상위 폴더의 .env에서 V-World 키를 읽습니다.
for env_dir in (cwd, *cwd.parents):
    env_path = env_dir / ".env"
    if env_path.exists():
        load_dotenv(env_path, override=False)
        break
vworld_key = os.getenv("V-WORLD_KEY")
if not vworld_key:
    raise ValueError(".env에 V-WORLD_KEY가 설정되어 있는지 확인하세요.")

# 2026년 9월 행정동별 시간대 생활인구와 월말 주민등록인구
living = pd.read_csv(
    data_dir / "busan_living_population_202609.csv",
    dtype={"행정동코드": "string"},
)
resident = pd.read_csv(
    data_dir / "busan_resident_population_202609.csv",
    dtype={"행정기관코드": "string"},
)
living["행정동코드"] = living["행정동코드"].str.strip().str.zfill(10)
resident["행정동코드"] = resident["행정기관코드"].str.strip().str.zfill(10)

if not living["기준년월"].astype(str).eq("202609").all():
    raise ValueError("생활인구 파일에 2026년 9월 외 자료가 포함되어 있습니다.")
resident = resident.loc[
    resident["기준연월"].astype(str).eq("2026-09-30")
    & resident["시도명"].eq("부산광역시")
].copy()

pop_cols = ["주거인구_일평균", "직장인구_일평균", "방문인구_일평균"]
for col in pop_cols:
    living[col] = pd.to_numeric(living[col], errors="coerce")
living["시간대_생활인구"] = living[pop_cols].sum(axis=1, min_count=3)
records_per_dong = living.groupby("행정동코드").size()
if not records_per_dong.eq(24).all():
    raise ValueError("행정동별 시간대 자료가 24개씩 있는지 확인하세요.")

# 시간대별 값을 합산하지 않고 평균해 하루 인구를 중복 집계하지 않습니다.
living_dong = (
    living.groupby("행정동코드", as_index=False)["시간대_생활인구"]
    .mean()
    .rename(columns={"시간대_생활인구": "생활인구_24시간평균"})
)
resident["주민등록인구"] = pd.to_numeric(resident["전체 당월인구수"], errors="coerce")
resident_dong = resident[["행정동코드", "주민등록인구"]].copy()

boundary = gpd.read_file(data_dir / "busan_hjd_20250101.geojson")
if boundary.crs is None:
    boundary = boundary.set_crs("EPSG:4326")
boundary = boundary.to_crs("EPSG:4326")
boundary["행정동코드"] = (
    boundary["adm_cd2"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(10)
)
boundary["행정동명"] = boundary["adm_nm"]
map_data = boundary.merge(living_dong, on="행정동코드", how="left").merge(
    resident_dong, on="행정동코드", how="left"
)
if len(map_data) != 206 or map_data[
    ["생활인구_24시간평균", "주민등록인구"]
].isna().any().any():
    raise ValueError("생활인구·주민등록인구·경계 코드 결합 결과를 확인하세요.")

map_data["생활인구_주민등록인구비"] = (
    map_data["생활인구_24시간평균"] / map_data["주민등록인구"]
)
labels = ["0.50 미만", "0.50–0.79", "0.80–0.99", "1.00–1.49", "1.50 이상"]
bins = [-np.inf, 0.5, 0.8, 1.0, 1.5, np.inf]
map_data["비율_범주"] = pd.cut(
    map_data["생활인구_주민등록인구비"],
    bins=bins,
    labels=labels,
    right=False,
).astype(str)
colors = {
    "0.50 미만": "#2166ac",
    "0.50–0.79": "#67a9cf",
    "0.80–0.99": "#d1e5f0",
    "1.00–1.49": "#f4a582",
    "1.50 이상": "#b2182b",
}

center_5179 = map_data.to_crs(epsg=5179).geometry.unary_union.centroid
center_4326 = gpd.GeoSeries([center_5179], crs=5179).to_crs(epsg=4326).iloc[0]
m = folium.Map(
    location=[center_4326.y, center_4326.x],
    zoom_start=10,
    tiles=None,
    control_scale=True,
)
folium.TileLayer(
    tiles=f"https://api.vworld.kr/req/wmts/1.0.0/{vworld_key}/Base/{{z}}/{{y}}/{{x}}.png",
    attr="© V-World",
    name="V-World 기본지도",
    max_zoom=19,
    control=False,
).add_to(m)

def style_feature(feature):
    category = feature["properties"].get("비율_범주")
    return {
        "fillColor": colors.get(category, "#bdbdbd"),
        "color": "#555555",
        "weight": 0.6,
        "fillOpacity": 0.78,
    }

folium.GeoJson(
    data=map_data.to_json(),
    name="생활인구 / 주민등록인구 비율",
    style_function=style_feature,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "행정동명",
            "생활인구_24시간평균",
            "주민등록인구",
            "생활인구_주민등록인구비",
            "비율_범주",
        ],
        aliases=[
            "행정동",
            "생활인구(24시간 평균)",
            "주민등록인구(9월 말)",
            "비율",
            "범주",
        ],
        localize=True,
        sticky=True,
    ),
).add_to(m)

legend_rows = "".join(
    f'<div style="display:flex;align-items:center;gap:7px;margin:3px 0;">'
    f'<span style="width:15px;height:13px;background:{colors[label]};border:1px solid #999"></span>{label}</div>'
    for label in labels
)
legend_html = (
    '<div style="position:fixed;z-index:9999;right:22px;bottom:28px;background:white;'
    'padding:10px 13px;border-radius:6px;box-shadow:0 1px 6px #0003;font-size:12px;">'
    '<b>생활인구 / 주민등록인구</b>' + legend_rows + '</div>'
)
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False).add_to(m)

# 미리보기 HTML에는 키가 렌더링되므로 .gitignore로 제외합니다.
preview_path = project_dir / "preview_202609.html"
m.save(str(preview_path))
display(IFrame(src=preview_path.name, width="100%", height=760))
